# Predicción de Abandono Escolar y Éxito Académico
## Modelo: Red Neuronal Artificial (ANN)
**Dataset:** Predict Students' Dropout and Academic Success — UCI ML Repository (ID 697)

In [ ]:
import pandas as pd

In [ ]:
students = pd.read_csv('students.csv')

In [ ]:
students.head()

In [ ]:
# Ver cuántas filas y columnas tiene el dataset
students.shape

In [ ]:
# Eliminar filas con valores nulos
students = students.dropna()
students.shape

In [ ]:
# Separar variables de entrada (X) y variable objetivo (y)
# Target: Dropout=0, Enrolled=1, Graduate=2
from sklearn.preprocessing import LabelEncoder
import pickle

CLASS_NAMES = ['Dropout', 'Enrolled', 'Graduate']
X = students.drop('Target', axis=1)
le = LabelEncoder()
le.fit(CLASS_NAMES)  # Orden fijo
y = le.transform(students['Target'])

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('LabelEncoder guardado')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [ ]:
# StandardScaler: normaliza los datos para que todas las columnas
# tengan la misma escala. Esto es importante para la Red Neuronal
# porque si una columna tiene valores muy grandes y otra muy pequeños,
# la red no aprende bien.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # ajusta y transforma el train
X_test_sc  = scaler.transform(X_test)       # solo transforma el test

# Guardar el scaler (lo necesita app.py para normalizar datos nuevos)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Scaler guardado')

In [ ]:
# Red Neuronal Artificial con 3 capas ocultas: 128, 64 y 32 neuronas
from sklearn.neural_network import MLPClassifier

ann = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
ann.fit(X_train_sc, y_train)

In [ ]:
y_pred = ann.predict(X_test_sc)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=CLASS_NAMES)
disp.plot(cmap='Reds')
plt.title('Matriz de Confusión - Red Neuronal Artificial (ANN)')
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Curva de pérdida durante el entrenamiento
plt.figure(figsize=(8, 4))
plt.plot(ann.loss_curve_, color='red')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.title('Curva de Entrenamiento - ANN (Students Dataset)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Guardar el modelo ANN entrenado para usarlo en app.py
with open('ann_model.pkl', 'wb') as f:
    pickle.dump(ann, f)

print('Modelo ANN guardado como ann_model.pkl')